# 04 · Correlation - Gast<->pSom antagonism along the hub

Within the **Mixed hub**, bin cells along pseudotime and measure the
module-vs-module Pearson correlation of the per-gene correlation vectors on
**Gast XOR pSom** genes, smoothed over the neighbour graph. A negative value is
the signature of mutual exclusivity - one embryonic program expressed at the
expense of the other as cells leave the hub. Deliverable: `correlation.pdf`,
`colorbars/R_colorbar.pdf`.

In [1]:
import os, sys, glob
os.environ['R_HOME'] = sys.exec_prefix + '/lib/R/'
import numpy as np, pandas as pd
import scanpy as sc, scanpy.external as sce, scFates as scf
import matplotlib.pyplot as plt
sc.settings.verbosity = 1

In [2]:
# cwd = <root>/cancer/vertebrate/CCCA/Claude/<Category>/<Dataset>
name = os.path.basename(os.getcwd())                    # dataset (folder minus 'Data_')
cat  = os.path.basename(os.path.dirname(os.getcwd()))   # Category partition
pth  = os.path.join(*([os.pardir]*6))                   # -> MesenCoder root
pth_data = os.path.join(pth, 'data')
sys.path.append(os.path.join(pth, 'cancer', 'vertebrate', 'CCCA'))
from scripts.load import load_CCCA_adata

pth_in  = os.path.join(pth_data, 'unzip', 'CCCA', 'Data_' + name)        # raw input
pth_out = os.path.join(pth_data, 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)  # h5ads
pth_fig = os.path.join(pth, 'figures', 'cancer', 'vertebrate', 'CCCA', 'Claude', cat, name)
pth_cb  = os.path.join(pth_fig, 'colorbars')
os.makedirs(pth_out, exist_ok=True); os.makedirs(pth_cb, exist_ok=True)

In [3]:
from scipy.stats import pearsonr

In [4]:
adata = sc.read_h5ad(os.path.join(pth_out, 'signatures.h5ad'))
sc.pl.umap(adata, color=['celltype','t'], size=120)

### Gast XOR pSom genes on the hub

In [5]:
feat = os.path.join(pth_data, 'features', 'biomart')
gast = set(pd.read_csv(os.path.join(feat,'GSE162534.csv')).hsapiens)
psom = set(pd.read_csv(os.path.join(feat,'GSE229103.csv')).hsapiens)
g = pd.Series(list(gast ^ psom)); g = g.loc[g.isin(adata.var_names)]
hub = adata[adata.obs.celltype=='Mixed', g].copy()
print('hub cells:', hub.n_obs, '| XOR genes:', g.size)

hub cells: 466 | XOR genes: 120


### Windowed module correlation, smoothed over the graph

In [6]:
n_cells = 100
n_bins = max(1, hub.n_obs // n_cells)
tbin = pd.cut(hub.obs['t'], bins=n_bins)
hub.obs['t_bin'] = tbin
r_dict = {}
for b, sdf in hub.obs.groupby('t_bin', observed=True):
    X = pd.DataFrame(hub[sdf.index].X.toarray(), index=sdf.index, columns=g)
    X = X.loc[:, X.nunique(0) > 1]
    a = pearsonr(X, sdf['GSE162534'].values[:,None], axis=0).statistic
    b2 = pearsonr(X, sdf['GSE229103'].values[:,None], axis=0).statistic
    r_dict[b] = np.corrcoef(a, b2)[0,1]
A = hub.obsp['connectivities']
Rv = tbin.map(r_dict).astype(float)
hub.obs['correlation'] = (A @ Rv) / A.sum(axis=1).A1
print('mean smoothed hub correlation r =', round(float(hub.obs['correlation'].mean()), 3))

mean smoothed hub correlation r = -0.224


### Module-correlation figure + colorbar (deliverable)

In [7]:
vlim = .5
fig, ax = plt.subplots(1, 1, figsize=(5,5))
sc.pl.umap(adata, size=180, alpha=.5, frameon=False, show=False, ax=ax)
sc.pl.umap(hub, color='correlation', cmap='Spectral', vmin=-vlim, vmax=vlim,
           size=180, alpha=.5, colorbar_loc=None, frameon=False, show=False, ax=ax)
sc.pl.umap(adata, color='celltype', size=180, alpha=0, legend_loc='on data',
           legend_fontoutline=1, frameon=False, show=False, ax=ax)
ax.invert_yaxis(); ax.invert_xaxis()
ax.set_title('Module Correlation', size=13)
fig.savefig(os.path.join(pth_fig,'correlation.pdf'), bbox_inches='tight')
fcb = plt.figure(figsize=(.66,1)); cax = fcb.add_axes([.35,.05,.3,.9])
cb = plt.colorbar(ax.collections[1], cax=cax)
cb.ax.tick_params(which='both', size=0); cb.set_label('Pearson\nCorrelation')
cb.ax.invert_yaxis()
ticks = np.asarray([-.5,-.25,0,.25,.5]); cb.set_ticks(ticks, labels=ticks)
cb.solids.set_alpha(1)
fcb.savefig(os.path.join(pth_cb,'R_colorbar.pdf'), bbox_inches='tight')

<cell>:12: UserWarning: Adding colorbar to a different Figure <Figure size 500x500 with 1 Axes> than <Figure size 66x100 with 1 Axes> which fig.colorbar is called on.
